# Allora Forge Builder Kit v2.0
## Topic 69 - Bitcoin Price Prediction Walkthrough

This walkthrough demonstrates 24-hour Bitcoin price prediction using the Allora ML Workflow Kit with base features and LightGBM.

---


## Installation

Install required dependencies and the Allora Forge Builder Kit from GitHub:


In [ ]:
# Install dependencies
!pip install numpy pandas scikit-learn lightgbm cloudpickle matplotlib

# Install Allora Forge Builder Kit from GitHub
!pip install git+https://github.com/allora-network/allora-forge-builder-kit.git


## Imports


In [1]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta, timezone
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMRegressor
import cloudpickle
from allora_forge_builder_kit import AlloraMLWorkflow, PerformanceEvaluator


## Experiment Configuration

Configure all experimental parameters in one place for easy tuning:


In [2]:
# Data Configuration
TICKERS = ["btcusd"]
DAYS_OF_HISTORY = 500
INTERVAL = "1h"

# Feature Configuration
NUMBER_OF_INPUT_BARS = 48  # Number of hourly bars for input features
TARGET_BARS = 24           # Predict 24 bars (hours) ahead

# Cross-Validation Configuration
N_SPLITS = 3               # Number of CV folds
MAX_TRAIN_SIZE = 100_000_000  # Maximum training samples per fold

# Model Configuration
N_ESTIMATORS_MAX = 500    # Train with max trees, evaluate at checkpoints
N_ESTIMATORS_CHECKPOINTS = [100, 300, 500]
LEARNING_RATES = [0.01, 0.05, 0.1]
MAX_DEPTHS = [3, 5, 7]
NUM_LEAVES = [15, 31, 63]


## Step 1: Initialize Workflow

Set up the Allora ML Workflow with your API key:


In [3]:
print("="*80)
print("Allora Forge Builder Kit v2.0 - Topic 69 Walkthrough")
print("="*80)

print("\n[1/6] Initializing workflow...")

# Read Allora API key
api_key_path = '.allora_api_key'
with open(api_key_path, 'r') as f:
    api_key = f.read().strip()

workflow = AlloraMLWorkflow(
    tickers=TICKERS,
    number_of_input_bars=NUMBER_OF_INPUT_BARS,
    target_bars=TARGET_BARS,
    interval=INTERVAL,
    data_source="allora",
    api_key=api_key
)

print(f"✅ Workflow initialized")
print(f"   Assets: {TICKERS} | Interval: {INTERVAL}")
print(f"   Input: {NUMBER_OF_INPUT_BARS} bars → Features: {NUMBER_OF_INPUT_BARS*5}")
print(f"   Target: {TARGET_BARS} bars ahead")


Allora Forge Builder Kit v2.0 - Topic 69 Walkthrough

[1/6] Initializing workflow...
✅ Workflow initialized
   Assets: ['btcusd'] | Interval: 1h
   Input: 48 bars → Features: 240
   Target: 24 bars ahead


## Step 2: Backfill Historical Data

Download and cache historical price data:


In [4]:
print(f"\n[2/6] Backfilling {DAYS_OF_HISTORY} days of historical data...")

start_date = datetime.now(timezone.utc) - timedelta(days=DAYS_OF_HISTORY)
workflow.backfill(start=start_date)
print("✅ Backfill complete")



[2/6] Backfilling 500 days of historical data...
[workflow] Backfilling ['btcusd'] 2024-06-18 18:34:55.214085+00:00 → now
[cleanup] Checking 1001 parquet files for corruption...
[cleanup] All 1001 parquet files are valid
[Allora backfill-missing] Checking btcusd 2024-06-18 18:34:55.214085+00:00 → 2025-10-31 18:34:56.093504+00:00
[Allora backfill-missing] Found 1001 parquet files for btcusd
[Allora backfill-missing] Date range: 2023-02-01 to 2025-10-28
[Allora backfill-missing] Complete days (1440 bars): 988
[Allora backfill-missing] btcusd: 9 incomplete days across 5 months, backfilling 2024-07-27 → 2025-04-05
[Allora backfill-missing] btcusd: backfilling 1 days in 2024-07
[Allora backfill] btcusd: 2024-07-01 00:00:00+00:00 → 2024-07-31 23:59:59.999999+00:00
[Allora backfill] btcusd: downloading 1 monthly buckets (filtered from 15)
[Allora backfill] btcusd: fetching API data from 2024-07-31 (1 days, max 6 pages)
[Allora backfill] btcusd: filtered to 1 incomplete days, writing 1427 bar

## Step 3: Extract & Engineer Features

Get base features and add log returns over multiple horizons:


In [6]:
print("\n[3/6] Extracting and engineering features...")

df_all = workflow.get_full_feature_target_dataframe(start_date=start_date).reset_index()

# Feature Engineering: Add log returns to base features
# For detailed TA indicators and visualizations, see: feature_engineering_example.py

def engineer_returns(row):
    """Add log return features over multiple horizons (no data leakage - same row only)"""
    # NOTE: Base features are already normalized (z-scored) by the workflow
    closes = np.array([row[f'feature_close_{i}'] for i in range(NUMBER_OF_INPUT_BARS)])
    
    # Log returns over different time horizons
    returns = {}
    returns['log_return_1h'] = np.log(closes[-1] + 1e-8) - np.log(closes[-2] + 1e-8) if NUMBER_OF_INPUT_BARS >= 2 else 0
    returns['log_return_6h'] = np.log(closes[-1] + 1e-8) - np.log(closes[-7] + 1e-8) if NUMBER_OF_INPUT_BARS >= 7 else 0
    returns['log_return_12h'] = np.log(closes[-1] + 1e-8) - np.log(closes[-13] + 1e-8) if NUMBER_OF_INPUT_BARS >= 13 else 0
    returns['log_return_24h'] = np.log(closes[-1] + 1e-8) - np.log(closes[-25] + 1e-8) if NUMBER_OF_INPUT_BARS >= 25 else 0
    
    return pd.Series(returns)

# Get base features
base_feature_cols = [col for col in df_all.columns if col.startswith('feature_')]

# Apply feature engineering
print("   Engineering log return features...")
engineered_features = df_all.apply(engineer_returns, axis=1)
df_all = pd.concat([df_all, engineered_features], axis=1)

# Use base features + engineered returns
feature_cols = base_feature_cols + list(engineered_features.columns)
df_all = df_all.dropna(subset=feature_cols + ['target'])

print(f"✅ Dataset: {len(df_all):,} samples ({df_all['open_time'].min().date()} to {df_all['open_time'].max().date()})")
print(f"   Features: {len(base_feature_cols)} base + {len(engineered_features.columns)} returns = {len(feature_cols)} total")
print(f"   📚 See feature_engineering_example.py for more TA indicators")



[3/6] Extracting and engineering features...
[workflow] Loading data
[workflow] Processing btcusd (719526 rows)
   Engineering log return features...
✅ Dataset: 11,924 samples (2024-06-20 to 2025-10-30)
   Features: 240 base + 4 returns = 244 total
   📚 See feature_engineering_example.py for more TA indicators


## Step 4: Define Time Series Splits

Walk-forward cross-validation through time, simulating period retraining with no data leakage.

In [7]:

# Setup time series cross-validation
tscv = TimeSeriesSplit(
    n_splits=N_SPLITS, 
    gap=TARGET_BARS, 
    max_train_size=MAX_TRAIN_SIZE
)

print(f"✅ Walk-forward CV: {N_SPLITS} splits, {TARGET_BARS}-bar embargo")
for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(df_all)):
    print(f"   Fold {fold_idx+1}: Train={len(train_idx):,}, Test={len(test_idx):,}")


✅ Walk-forward CV: 3 splits, 24-bar embargo
   Fold 1: Train=2,957, Test=2,981
   Fold 2: Train=5,938, Test=2,981
   Fold 3: Train=8,919, Test=2,981


## Step 4: Grid Search with Walk-Forward CV

Train LightGBM models with different hyperparameters and evaluate:


In [8]:
print("\n[4/6] Running grid search...")

results = []
evaluator = PerformanceEvaluator()
config_num = 0

for lr in LEARNING_RATES:
    for depth in MAX_DEPTHS:
        for leaves in NUM_LEAVES:
            
            # Train once with max trees, evaluate at checkpoints
            fold_models = []
            for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(df_all)):
                X_train = df_all.iloc[train_idx][feature_cols]
                y_train = df_all.iloc[train_idx]['target']
                
                lgb = LGBMRegressor(
                    n_estimators=N_ESTIMATORS_MAX,
                    learning_rate=lr,
                    max_depth=depth,
                    num_leaves=leaves,
                    random_state=42,
                    verbose=-1
                )
                lgb.fit(X_train, y_train)
                fold_models.append((lgb, test_idx))
            
            # Evaluate at tree count checkpoints
            for n_est in N_ESTIMATORS_CHECKPOINTS:
                config_num += 1
                df_all['pred'] = np.nan
                
                # Generate predictions using first n_est trees
                for lgb, test_idx in fold_models:
                    X_test = df_all.iloc[test_idx][feature_cols]
                    preds = lgb.predict(X_test, num_iteration=n_est)
                    df_all.iloc[test_idx, df_all.columns.get_loc('pred')] = preds
                
                # Evaluate
                valid_mask = ~df_all['pred'].isna()
                metrics = evaluator.evaluate(
                    y_true=df_all.loc[valid_mask, 'target'],
                    y_pred=df_all.loc[valid_mask, 'pred']
                )
                
                # Store results
                results.append({
                    'config_num': config_num,
                    'n_estimators': n_est,
                    'learning_rate': lr,
                    'max_depth': depth,
                    'num_leaves': leaves,
                    'predictions': df_all['pred'].copy(),
                    **metrics
                })
                
                print(f"   [{config_num:2d}] n={n_est:4d}, lr={lr:.2f}, d={depth}, l={leaves:2d} → "
                      f"{metrics['num_passed']}/8 ({metrics['score']:.1%} - {metrics['grade']})")

# Analyze results
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'predictions'} for r in results])
results_df = results_df.sort_values(['num_passed', 'score'], ascending=[False, False])

print(f"\n✅ Tested {len(results)} configurations")
print(f"\n   Top 5 models:")
top5_cols = ['config_num', 'n_estimators', 'learning_rate', 'max_depth', 'num_leaves', 'num_passed', 'score']
print(results_df[top5_cols].head().to_string(index=False))

# Select best model
best_result = results[results_df.iloc[0]['config_num'] - 1]
best_params = {k: best_result[k] for k in ['n_estimators', 'learning_rate', 'max_depth', 'num_leaves']}

print(f"\n✅ Best: Config #{best_result['config_num']}")
print(f"   {best_result['num_passed']}/8 metrics ({best_result['score']:.1%}) | "
      f"n={best_params['n_estimators']}, lr={best_params['learning_rate']}, d={best_params['max_depth']}, l={best_params['num_leaves']}")



[4/6] Running grid search...
   [ 1] n= 100, lr=0.01, d=3, l=15 → 4/8 (50.0% - C)
   [ 2] n= 300, lr=0.01, d=3, l=15 → 3/8 (37.5% - D)
   [ 3] n= 500, lr=0.01, d=3, l=15 → 3/8 (37.5% - D)
   [ 4] n= 100, lr=0.01, d=3, l=31 → 4/8 (50.0% - C)
   [ 5] n= 300, lr=0.01, d=3, l=31 → 3/8 (37.5% - D)
   [ 6] n= 500, lr=0.01, d=3, l=31 → 3/8 (37.5% - D)
   [ 7] n= 100, lr=0.01, d=3, l=63 → 4/8 (50.0% - C)
   [ 8] n= 300, lr=0.01, d=3, l=63 → 3/8 (37.5% - D)
   [ 9] n= 500, lr=0.01, d=3, l=63 → 3/8 (37.5% - D)
   [10] n= 100, lr=0.01, d=5, l=15 → 4/8 (50.0% - C)
   [11] n= 300, lr=0.01, d=5, l=15 → 3/8 (37.5% - D)
   [12] n= 500, lr=0.01, d=5, l=15 → 4/8 (50.0% - C)
   [13] n= 100, lr=0.01, d=5, l=31 → 4/8 (50.0% - C)
   [14] n= 300, lr=0.01, d=5, l=31 → 3/8 (37.5% - D)
   [15] n= 500, lr=0.01, d=5, l=31 → 4/8 (50.0% - C)
   [16] n= 100, lr=0.01, d=5, l=63 → 4/8 (50.0% - C)
   [17] n= 300, lr=0.01, d=5, l=63 → 3/8 (37.5% - D)
   [18] n= 500, lr=0.01, d=5, l=63 → 4/8 (50.0% - C)
   [19] n= 100, 

## Step 5: Evaluate Best Model

Review detailed performance metrics:


In [9]:
print("\n[5/6] Detailed evaluation...")
print("="*80)
evaluator.print_report(best_result, detailed=False)
print("="*80)



[5/6] Detailed evaluation...
PERFORMANCE EVALUATION REPORT

📊 OVERALL PERFORMANCE: C (4/8 metrics passed)
   Performance Score: 50.00%

🎯 PRIMARY METRICS (8 Core Metrics)

1. Directional Accuracy:
   Value: 0.5340  ❌ FAIL
   Threshold: ≥ 0.55
   Correct: 4776/8943 predictions

2. DA Confidence Interval (95%):
   Lower: 0.5237  ✅ PASS
   Upper: 0.5444
   Threshold: Lower ≥ 0.52

3. DA Statistical Significance:
   p-value: 0.0000  ✅ PASS
   Threshold: < 0.05

4. Pearson Correlation:
   r: 0.1113  ✅ PASS
   Threshold: ≥ 0.05

5. Pearson Statistical Significance:
   p-value: 0.0000  ✅ PASS
   Threshold: < 0.05

6. WRMSE Improvement:
   Improvement: 0.0144 (1.44%)  ❌ FAIL
   Threshold: ≥ 0.1 (10%)
   Model WRMSE: 0.038197
   Baseline WRMSE: 0.038757

7. ZPTAE Improvement:
   Improvement: 0.0131 (1.31%)  ❌ FAIL
   Threshold: ≥ 0.2 (20%)
   Model ZPTAE: 0.672702
   Baseline ZPTAE: 0.681616

8. Log Aspect Ratio:
   Value: -0.8696  ❌ FAIL
   Threshold: |value| < 0.5
   Std(true): 0.022808
   S

## Step 6: Train Production Model

Train final model on all data and save prediction function:


In [10]:
print("\n[6/6] Training production model...")

final_model = LGBMRegressor(
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    num_leaves=best_params['num_leaves'],
    random_state=42,
    verbose=-1
)
final_model.fit(df_all[feature_cols], df_all['target'])
print(f"✅ Final model trained on {len(df_all):,} samples")

def predict(nonce: int = None) -> float:
    """
    Predict Bitcoin price 24 hours into the future.
    
    Args:
        nonce: Block nonce from Allora SDK (unused)
    
    Returns:
        float: Predicted BTC price in USD
    """
    # Get live features from workflow
    live_row = workflow.get_live_features(ticker=TICKERS[0])
    
    if live_row is None or len(live_row) == 0:
        raise ValueError("Could not get live features")
    
    # Engineer return features from live data (same as training)
    live_returns = engineer_returns(live_row.iloc[0])
    
    # Combine base features + engineered returns
    live_features = pd.concat([live_row[base_feature_cols].iloc[0], live_returns])
    
    # Get current price
    now = datetime.now(timezone.utc)
    recent_start = now - timedelta(hours=1)
    raw_data = workflow.load_raw(start=recent_start, end=now)
    current_price = raw_data["close"].iloc[-1] if len(raw_data) > 0 else 0.0
    
    # Predict log return
    predicted_log_return = final_model.predict(live_features[feature_cols].values.reshape(1, -1))[0]
    
    # Convert log return to price
    predicted_price = current_price * np.exp(predicted_log_return)
    
    print(f"\nLive Prediction: ${predicted_price:,.2f} ({predicted_log_return:+.4f} log return)")
    
    return float(predicted_price)

# Test and save
print("\n🧪 Testing prediction...")
test_prediction = predict()

with open("predict.pkl", "wb") as f:
    cloudpickle.dump(predict, f)

# Summary
print("\n" + "="*80)
print("COMPLETE!")
print("="*80)
print(f"✅ {len(feature_cols)} features | {best_result['num_passed']}/8 metrics ({best_result['score']:.1%})")
print(f"✅ Saved to predict.pkl")
print("="*80)
print("\n🚀 Deploy: python deploy_worker.py")



[6/6] Training production model...
✅ Final model trained on 11,924 samples

🧪 Testing prediction...

Live Prediction: $109,060.08 (+0.0009 log return)

COMPLETE!
✅ 244 features | 4/8 metrics (50.0%)
✅ Saved to predict.pkl

🚀 Deploy: python deploy_worker.py


/home/jefferythewind/anaconda3/envs/ml311_dev/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


---

## Next Steps

1. **Explore Feature Engineering**: Check out `feature_engineering_example.py` for visualizations and more TA indicators (SMAs, MACD, etc.)
2. **Deploy to Allora Network**: Run `python deploy_worker.py` to start submitting predictions
3. **Experiment**: Modify the configuration parameters at the top to test different setups

---
